# GANForge -- SRGAN (Standalone Notebook)

**Single-image super-resolution**

This is a **self-contained extract** of the SRGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="9"></a>
## 9. SRGAN -- Single-image super-resolution

### 9.1 Problem Statement & Why this GAN

Ledig et al. (2017) tackled **single-image super-resolution**: given only
a low-resolution image, hallucinate a plausible 4x higher-resolution
version. Pure pixel-MSE-trained super-resolution networks (pre-SRGAN)
produce blurry outputs, because MSE is minimized by the *average* of all
plausible high-frequency completions -- exactly the wrong objective when
"plausible-looking detail" matters more than "literally correct per
-pixel value" (a genuinely different problem from every earlier section:
here the input is a full image, not noise, and there is a well-defined
"correct-ish" target). SRGAN's fix: train with a **perceptual loss**
(distance in a pretrained VGG's feature space, not pixel space) *plus* an
**adversarial loss**, so the generator is rewarded for producing detail
that *looks* real, not detail that minimizes pixel-wise error.

### 9.2 Data Collection & Preparation

In [ ]:
SRGAN_HR_SIZE = 96          # STL-10's native resolution -- convenient, no cropping needed
SRGAN_SCALE = 4
SRGAN_LR_SIZE = SRGAN_HR_SIZE // SRGAN_SCALE   # 24x24 input -> 96x96 output
SRGAN_BATCH_SIZE = 16
SRGAN_EPOCHS = 6

hr_transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

try:
    srgan_hr_base = torchvision.datasets.STL10(root="data", split="train", download=True, transform=hr_transform)
    print(f"Loaded real STL-10 (as HR source; production setups should use DIV2K): {len(srgan_hr_base)} images")
except Exception as e:
    print(f"STL-10 download failed ({e}); falling back to synthetic textures.")
    rng = np.random.default_rng(SEED)
    tex = torch.rand(500, 3, SRGAN_HR_SIZE, SRGAN_HR_SIZE) * 2 - 1
    srgan_hr_base = TensorDataset(tex, torch.zeros(500, dtype=torch.long))

class SRDataset(Dataset):
    '''Builds (LR, HR) PAIRS on the fly: HR is the original image, LR is a
    4x bicubic-downsampled version. Bicubic downsampling is the SRGAN
    paper's degradation model -- swap this for a learned/realistic
    degradation model (blur+noise+JPEG) for production-quality real-world
    super-resolution.'''
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        hr, _ = self.base[i]
        lr = F.interpolate(hr.unsqueeze(0), size=SRGAN_LR_SIZE, mode="bicubic", align_corners=False, antialias=True)
        return lr.squeeze(0).clamp(-1, 1), hr

srgan_dataset = SRDataset(srgan_hr_base)
srgan_loader = DataLoader(srgan_dataset, batch_size=SRGAN_BATCH_SIZE, shuffle=True, drop_last=True)
lr_batch, hr_batch = next(iter(srgan_loader))
show_images(F.interpolate(lr_batch[:8], size=SRGAN_HR_SIZE, mode="nearest"), title="LR inputs (nearest-upsampled for display)", nrow=8, figsize=(12, 2))
show_images(hr_batch[:8], title="HR targets", nrow=8, figsize=(12, 2))


### 9.3 Model Architecture

In [ ]:
class SRGAN_ResBlock(nn.Module):
    '''BN + PReLU residual block operating entirely at LOW resolution
    (before upsampling) -- keeps the expensive deep processing cheap, with
    all detail-adding upsampling deferred to the end via PixelShuffle.'''
    def __init__(self, ch=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.PReLU(),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return x + self.block(x)


class PixelShuffleUpsample(nn.Module):
    '''Sub-pixel convolution (Shi et al., 2016): convolve to 4x the target
    channels, then rearrange (PixelShuffle) into 2x spatial resolution.
    This produces far fewer checkerboard artifacts than transposed
    convolution, which is why SRGAN (and most later SR GANs) use it.'''
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch * 4, 3, 1, 1)
        self.shuffle = nn.PixelShuffle(2)
        self.act = nn.PReLU()

    def forward(self, x):
        return self.act(self.shuffle(self.conv(x)))


class SRGAN_Generator(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, n_res_blocks=8, scale=SRGAN_SCALE):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(in_ch, base_ch, 9, 1, 4), nn.PReLU())
        self.res_blocks = nn.Sequential(*[SRGAN_ResBlock(base_ch) for _ in range(n_res_blocks)])
        self.mid = nn.Sequential(nn.Conv2d(base_ch, base_ch, 3, 1, 1), nn.BatchNorm2d(base_ch))
        n_upsamples = int(math.log2(scale))
        self.upsample = nn.Sequential(*[PixelShuffleUpsample(base_ch) for _ in range(n_upsamples)])
        self.exit = nn.Conv2d(base_ch, 3, 9, 1, 4)

    def forward(self, x):
        entry_feat = self.entry(x)
        res_feat = self.mid(self.res_blocks(entry_feat))
        feat = entry_feat + res_feat   # global residual skip AROUND all residual blocks (paper's key stabilizer for deep SR nets)
        feat = self.upsample(feat)
        return torch.tanh(self.exit(feat))


class SRGAN_Discriminator(nn.Module):
    '''VGG-style deep classifier at the HR resolution -- alternating
    stride-1/stride-2 conv blocks with growing channel count, judging
    "is this a real HR photo or an upsampled fake" globally (unlike
    CycleGAN's PatchGAN, a single global decision suits the SR task,
    since realism here is a property of the whole image's sharpness).'''
    def __init__(self, base_ch=64):
        super().__init__()
        def conv_bn_lrelu(in_c, out_c, stride, bn=True):
            layers = [nn.Conv2d(in_c, out_c, 3, stride, 1)]
            if bn:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers
        layers = []
        layers += conv_bn_lrelu(3, base_ch, 1, bn=False)
        layers += conv_bn_lrelu(base_ch, base_ch, 2)
        layers += conv_bn_lrelu(base_ch, base_ch * 2, 1)
        layers += conv_bn_lrelu(base_ch * 2, base_ch * 2, 2)
        layers += conv_bn_lrelu(base_ch * 2, base_ch * 4, 1)
        layers += conv_bn_lrelu(base_ch * 4, base_ch * 4, 2)
        layers += conv_bn_lrelu(base_ch * 4, base_ch * 8, 1)
        layers += conv_bn_lrelu(base_ch * 8, base_ch * 8, 2)
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(6), nn.Flatten(),
            nn.Linear(base_ch * 8 * 6 * 6, 1024), nn.LeakyReLU(0.2, True),
            nn.Linear(1024, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).view(-1)


G_srgan = SRGAN_Generator().to(DEVICE)
D_srgan = SRGAN_Discriminator().to(DEVICE).apply(weights_init)


### 9.4 Loss Functions & Optimizers

In [ ]:
def build_vgg_feature_extractor():
    '''Perceptual ("content") loss network: a pretrained VGG19's early
    conv-block features, frozen. Comparing SR/HR images in THIS feature
    space (rather than raw pixels) rewards texture/structure similarity
    the way a human perceives it, not literal per-pixel agreement.'''
    try:
        vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features[:36].to(DEVICE).eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        print("Loaded pretrained VGG19 for perceptual loss.")
        return vgg
    except Exception as e:
        print(f"Pretrained VGG19 unavailable ({e}); falling back to pixel-space MSE as the content loss.")
        return None

srgan_vgg = build_vgg_feature_extractor()
srgan_vgg_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
srgan_vgg_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

def srgan_content_loss(sr, hr):
    if srgan_vgg is None:
        return F.mse_loss(sr, hr)
    sr_n = (denorm(sr) - srgan_vgg_mean) / srgan_vgg_std
    hr_n = (denorm(hr) - srgan_vgg_mean) / srgan_vgg_std
    return F.mse_loss(srgan_vgg(sr_n), srgan_vgg(hr_n))

srgan_adv_criterion = nn.BCEWithLogitsLoss()
SRGAN_LAMBDA_ADV = 1e-3   # the paper's own weighting: adversarial term is intentionally tiny relative to content loss
srgan_opt_G = torch.optim.Adam(G_srgan.parameters(), lr=1e-4, betas=(0.9, 0.999))
srgan_opt_D = torch.optim.Adam(D_srgan.parameters(), lr=1e-4, betas=(0.9, 0.999))


### 9.5 Training Loop

In [ ]:
def psnr(sr, hr):
    '''Peak Signal-to-Noise Ratio in dB -- the standard quantitative metric
    for super-resolution (higher = closer to the ground-truth HR image).'''
    mse = F.mse_loss(denorm(sr), denorm(hr)).item()
    return 10 * math.log10(1.0 / max(mse, 1e-10))

srgan_history = []
for epoch in range(SRGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "PSNR": AvgMeter()}
    pbar = tqdm(srgan_loader, desc=f"[SRGAN] epoch {epoch+1}/{SRGAN_EPOCHS}")
    for lr, hr in pbar:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        b = lr.size(0)

        sr = G_srgan(lr)

        # ---- Discriminator ----
        D_srgan.zero_grad(set_to_none=True)
        loss_d = srgan_adv_criterion(D_srgan(hr), torch.ones(b, device=DEVICE)) + \
                 srgan_adv_criterion(D_srgan(sr.detach()), torch.zeros(b, device=DEVICE))
        loss_d.backward()
        srgan_opt_D.step()

        # ---- Generator: content (perceptual) + tiny adversarial term ----
        G_srgan.zero_grad(set_to_none=True)
        content = srgan_content_loss(sr, hr)
        adv = srgan_adv_criterion(D_srgan(sr), torch.ones(b, device=DEVICE))
        loss_g = content + SRGAN_LAMBDA_ADV * adv
        loss_g.backward()
        srgan_opt_G.step()

        meters["G"].update(loss_g.item()); meters["D"].update(loss_d.item()); meters["PSNR"].update(psnr(sr, hr))
        pbar.set_postfix(loss_G=f"{loss_g.item():.3f}", loss_D=f"{loss_d.item():.3f}", PSNR=f"{meters['PSNR'].avg:.1f}dB")

    srgan_history.append((meters["G"].avg, meters["D"].avg, meters["PSNR"].avg))
    with torch.no_grad():
        G_srgan.eval()
        sample_sr = G_srgan(lr[:4])
        G_srgan.train()
    show_images(torch.cat([F.interpolate(lr[:4], size=SRGAN_HR_SIZE, mode="nearest"), sample_sr, hr[:4]]),
                title=f"SRGAN epoch {epoch+1}: LR (nearest) / SR / HR", nrow=4)
    save_checkpoint({"G": G_srgan.state_dict(), "D": D_srgan.state_dict()}, f"{CHECKPOINT_DIR}/srgan_epoch{epoch+1}.pt")

print("SRGAN training complete.")


### 9.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def srgan_upscale(lr_images: torch.Tensor) -> torch.Tensor:
    '''Clean inference entry point: LR image(s) in ([-1,1], any small size),
    4x super-resolved image(s) out.'''
    G_srgan.eval()
    sr = G_srgan(lr_images.to(DEVICE))
    G_srgan.train()
    return sr

demo_lr = lr_batch[:4].to(DEVICE)
show_images(srgan_upscale(demo_lr), title="SRGAN inference: 4x super-resolved", nrow=4)


### 9.7 Visualization & Results

In [ ]:
gs, ds, psnrs = zip(*srgan_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gs, label="G"); axes[0].plot(ds, label="D"); axes[0].legend(); axes[0].set_title("Losses")
axes[1].plot(psnrs); axes[1].set_title("PSNR (dB) -- higher is better")
plt.show()

# Side-by-side: bicubic upsampling (a naive, non-learned baseline) vs SRGAN vs ground truth.
with torch.no_grad():
    lr4, hr4 = lr_batch[:4].to(DEVICE), hr_batch[:4].to(DEVICE)
    bicubic = F.interpolate(lr4, size=SRGAN_HR_SIZE, mode="bicubic", align_corners=False)
    sr4 = srgan_upscale(lr4)
fig, axes = plt.subplots(4, 3, figsize=(6, 8))
for row in range(4):
    for col, (img, name) in enumerate(zip([bicubic[row], sr4[row], hr4[row]], ["Bicubic", "SRGAN", "Ground truth"])):
        axes[row, col].imshow(denorm(img).permute(1, 2, 0).cpu()); axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(name)
plt.tight_layout(); plt.show()
print(f"Bicubic PSNR: {psnr(bicubic, hr4):.2f}dB | SRGAN PSNR: {psnr(sr4, hr4):.2f}dB")


### 9.8 User Input → Output Demo

In [ ]:
IMAGE_INDEX = 2   # which validation image to super-resolve

user_lr = lr_batch[IMAGE_INDEX:IMAGE_INDEX + 1]
user_sr = srgan_upscale(user_lr)
show_images(torch.cat([F.interpolate(user_lr, size=SRGAN_HR_SIZE, mode="nearest"), user_sr]),
            title="User demo: LR input (nearest-upsampled) vs SRGAN output", nrow=2, figsize=(4, 2))


### 9.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** "Lower pixel MSE" and "looks more realistic" are
  genuinely different objectives for super-resolution -- SRGAN explicitly
  trades a bit of the former for a lot of the latter, which is why SRGAN
  images often score *worse* on PSNR than a purely MSE-trained baseline
  while looking clearly sharper/more detailed to a human. Always report
  both a perceptual comparison AND PSNR/SSIM, never PSNR alone.
- ⚠️ **Warning.** SRGAN can hallucinate texture that is plausible but
  factually wrong (invented detail not present in the true HR image) --
  unacceptable for domains needing faithful reconstruction (medical
  imaging, forensics), fine for domains where "looks good" is the goal.

**This GAN solved the problem of: single-image super-resolution --
recovering plausible high-frequency detail from a low-resolution input,
optimized for perceptual realism rather than pixel-wise accuracy alone.**